In [1]:
from dotenv import load_dotenv
import os
import ibis
from ibis import _
import pandas as pd
import numpy as np
import time
import re

In [2]:
from dash import Dash, dcc, html, Input, Output, callback,dash_table
from dash import State, no_update
from dash.exceptions import PreventUpdate
import json
import dash_bootstrap_components as dbc
from dash import ctx

In [3]:
# load env

load_dotenv(override=True)
kwargs_con = {
    'host':os.getenv('db_host_'),
    'user':os.getenv('db_user_'),
    'port':os.getenv('db_port_'),
    'password':os.getenv('db_password_'),
    'database':os.getenv('db_database_'),
    'driver':os.getenv('db_driver_')
}

kwargs_tbl_name = {
    'tbl_pr_':os.getenv('db_table_province_'),
    'tbl_car_':os.getenv('db_table_car_'),
    'tbl_sp_':os.getenv('db_table_sp_')
}

In [4]:
class IbisMSSQLConnection:
    """Context Manager for connect to MSSQL through ibis"""
    
    def __init__(self, connection_params):
        """
        Initialize connection manager
        
        Args:
            connection_params: connection parameters
        """
        self.connection_params = connection_params
        self.connection= None
    
    def __enter__(self):
        """ __enter__ part of context"""
        try:
            self.connection = ibis.mssql.connect(**self.connection_params)
            # print('--- Connected ---')
            return self.connection
        except Exception as e:
            raise ConnectionError(f"Error in connecting to database(__enter__): {e}")
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """close part of context"""
        if self.connection:
            try:
                self.connection.disconnect()
                # print('--- Disconnected ---')
            except Exception as close_error:
                print(f"⚠️ Error in closing the connection (__exit__) : {close_error}")
            finally:
                self.connection = None
        
        # اگر خطایی در context رخ داده بود، آن را suppress نکن
        # return False = “خطا را به بیرون منتقل کن (propagate)”
        # return True = “خطا را سرکوب کن (suppress)”
        #برای connection managerها معمولاً return False مناسب‌تر است چون می‌خواهیم خطاهای واقعی برنامه دیده شوند.
        if exc_type is not None:
            print(f"خطای {exc_type.__name__}: {exc_val}")
        
            # observing traceback
            import traceback
            traceback.print_tb(exc_tb)
        return False


In [5]:
# try:
#     conn = ibis.mssql.connect(**kwargs_con)
#     print('--- Connected ---')
# except Exception as e:
#     raise ConnectionError(f"Error in connecting to database(__enter__): {e}")

In [6]:
def load_list_tables():
    try:
        with IbisMSSQLConnection(kwargs_con) as con:
            return list(con.list_tables())
    except Exception as e:
        print(e)
        return []

def load_data(tbl_,clmns_,limit_):
    with IbisMSSQLConnection(kwargs_con) as con:
        tbl = con.table(tbl_)
        data = (
            tbl.limit(limit_)
        
        ).execute()
    return data

In [8]:
app = Dash(__name__ ,
    external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container(
    [
        dbc.Row(
            [
                dbc.Col(
                    dcc.Dropdown(
                        id="Id_load_list_tables",
                        multi=False,
                    ),
                    width=3,
                ),
                dbc.Col(
                    dcc.Input(
                        id="Id_limit",
                        type="number",
                        min=1,
                        max=1000,
                        step=1,
                        value=5,
                    ),
                    width=2,
                ),
                dbc.Col(
                    dcc.Dropdown(
                        id="Id_load_list_tables_2",
                        multi=False,
                    ),
                    width=3,
                ),
                dbc.Col(
                    dcc.Input(
                        id="Id_limit_2",
                        type="number",
                        min=1,
                        max=1000,
                        step=1,
                        value=5,
                    ),
                    width=2,
                ),
            ],
            className="mb-3",
        ),

        dbc.Row(
            [
                dbc.Col(
                    dash_table.DataTable(
                        id="Id_tbl",
                        export_format="csv",
                        export_headers="display",
                    ),
                    width=12,
                )
            ]
        ),

        dbc.Row(
            [               
                dbc.Col(
                    dash_table.DataTable(
                        id="Id_tbl_2",
                        export_format="csv",
                        export_headers="display",
                    ),
                    width=12,
                ),
            ]
        ),

        
    ],
    fluid=True,
)


@callback(
    Output('Id_load_list_tables', 'options'),
    Output('Id_load_list_tables', 'value'),
    Input('Id_load_list_tables', 'id')
)
def init_load_list_tables(id_):
    list_tbl = load_list_tables()
    options = [{'label': r, 'value': r} for r in list_tbl]
    # value = options[0]['value'] if options else None
    value = "FactContract"
    return options, value

@callback(
    Output('Id_load_list_tables_2', 'options'),
    Output('Id_load_list_tables_2', 'value'),
    Input('Id_load_list_tables_2', 'id')
)
def init_load_list_tables_2(id_):
    list_tbl = load_list_tables()
    options = [{'label': r, 'value': r} for r in list_tbl]
    # value = options[0]['value'] if options else None
    value = "FactDispatch"
    return options, value


@app.callback(
    Output("Id_tbl", "columns"),
    Output("Id_tbl", "data"),
    Input("Id_load_list_tables","value"),
    Input("Id_limit","value")
)
def init_data_tbl(tbl_,limit_):
    if not tbl_:
        return [], []
    
    df = load_data(tbl_,limit_)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")

@app.callback(
    Output("Id_tbl_2", "columns"),
    Output("Id_tbl_2", "data"),
    Input("Id_load_list_tables_2","value"),
    Input("Id_limit_2","value")
)
def init_data_tbl_2(tbl_,limit_):
    if not tbl_:
        return [], []
    
    df = load_data(tbl_,limit_)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")

   
if __name__ == '__main__':
    app.run(debug=True,port=9871)

In [2]:
def load_provinces():
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.select('provinceName').distinct().execute()
        distinct_pr = distinct_pr.provinceName.to_list()
    return distinct_pr

def load_cities(prs):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.filter(distinct_pr.provinceName.isin(prs))
        distinct_pr = distinct_pr.select('cityName').distinct().execute()
        distinct_pr = distinct_pr.cityName.to_list()
    return distinct_pr

def load_roadNames(prs,cities):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = (distinct_pr.filter(distinct_pr.provinceName.isin(prs))
            .filter(distinct_pr.cityName.isin(cities))
                      )
        distinct_pr = distinct_pr.select('sppRoadName').distinct().execute()
        distinct_pr = distinct_pr.sppRoadName
    return distinct_pr


def load_cameraIds(prs,cities,roadNames):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = (
            distinct_pr.filter(distinct_pr.provinceName.isin(prs))
            .filter(distinct_pr.cityName.isin(cities))
            .filter(distinct_pr.sppRoadName.isin(roadNames))
                      )
        distinct_pr = distinct_pr.select("cameraId").distinct().execute()
        distinct_pr = distinct_pr.cameraId
    return distinct_pr

def load_data(prs,cities,roadNames,cameraIds):
    with IbisPostgresConnection(kwargs_con) as con:
        tbl = con.table(kwargs_tbl_name['tbl_pr_'])
        data = (
            tbl.filter(_.provinceName.isin(prs))
            .filter(_.cityName.isin(cities))
            .filter(_.sppRoadName.isin(roadNames))
            .filter(_.cameraId.isin(cameraIds))
        ).execute()
    return data



app = Dash(__name__ ,
    external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dcc.Tabs([
        dcc.Tab(label='استان', children=[
            dcc.Dropdown(id='Id_province',multi=True)
        ]),
        dcc.Tab(label='شهرستان', children=[
            dcc.Dropdown(id='Id_city',multi=True)
        ]),
        dcc.Tab(label='محور', children=[
            dcc.Dropdown(id='Id_roadName',multi=True)
        ]),
        dcc.Tab(label='آیدی دوربین', children=[
            dcc.Dropdown(id='Id_cameraId',multi=True)
        ])
    ]),
   
    dash_table.DataTable(id="id_tbl" ,export_format="csv",    export_headers="display"),
    
])

@callback(
    Output('Id_province', 'options'),
    Output('Id_province', 'value'),
    Input('Id_province', 'id')
)
def init_provinces(_):
    df = load_provinces()
    options = [{'label': r, 'value': r} for r in df]
    value = options[0]['value'] if options else None
    return options, value


@callback(
    Output("Id_city", "options"),
    Output("Id_city", "value"),
    Input("Id_province", "value")
)
def init_city(prs):
    if not prs:
        return [], []
    ###### This line is very important
    if isinstance(prs, str):
        prs = [prs]
    ######
    df = load_cities(prs)
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]
    
    return options, value

@callback(
    Output('Id_roadName', 'options'),
    Output('Id_roadName', 'value'),
    Input('Id_province', 'value'),
    Input('Id_city', 'value')    
)
def init_roadName(prs,cities):
    if not prs or not cities:
        return [], []

    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]

    df = load_roadNames(prs, cities)
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]
    return options, value

@callback(
    Output('Id_cameraId', 'options'),
    Output('Id_cameraId', 'value'),
    Input('Id_province', 'value'),
    Input('Id_city', 'value'),
    Input('Id_roadName', 'value')
)
def init_cameraId(prs,cities,roadNames):
    if not prs or not cities or not roadNames :
        return [], []

    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]
    if isinstance(roadNames, str):
        roadNames = [roadNames]
    df = load_cameraIds(prs, cities,roadNames)    
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]    
    return options, value


@app.callback(
    Output("id_tbl", "columns"),
    Output("id_tbl", "data"),
    Input("Id_province","value"),
    Input("Id_city","value"),
    Input("Id_roadName","value"),
    Input("Id_cameraId","value")
)
def init_data(prs,cities,roadNames,cameraIds):
    if not cameraIds:
        return [], []
    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]
    if isinstance(roadNames, str):
        roadNames = [roadNames]
    if isinstance(cameraIds, str):
        roadNames = [cameraIds]

    df = load_data(prs,cities,roadNames,cameraIds)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")


    
if __name__ == '__main__':
    app.run(debug=True,port=9871)

In [170]:
tbl_FactContract.select(ibis.selectors.contains(["Year","Month"])).columns

('ShamsiYearMonth',)

In [171]:
tbl_FactContract.limit(5).execute()

,ContractID,ContractCode,ContractKindId,DispatchCount,ContractStatusId,MiladiCreateDate,ShamsiCreateDate,CreateTime,CreatorUserId,CreatorUserBranchId,...,GoodKindCustomer,ReceiverBranchId,LockerAuthCode,BunchId,SmsInternalTime,ResultBranchID,SenderLat,SenderLong,ChannelsId,Channelstitle
0,1,32071202582191128515,1,1,7,2025-08-02 18:13:53.623,14040511,1181353,31867,298,...,None,NaN,None,None,NaT,None,35.7485583333333,51.3163816666667,NaN,None
1,2,603422025925175366186,1,1,7,2025-09-25 17:54:54.273,14040703,1175454,38755,766,...,اسباب‌بازی و سرگرمی,507.0,None,None,NaT,None,38.05755194,46.36966897,NaN,None
2,3,3142920250930185538164,1,1,7,2025-08-30 18:55:38.627,14040608,1185538,5545,1887,...,چندکالایی / مارکت‌پلیس,NaN,None,None,NaT,None,35.6903833,51.2169663,7,وب سرویس سفارشات
3,4,112952025330103187288,1,1,7,2025-03-30 09:33:47.040,14040110,1093347,26556,505,...,None,NaN,None,None,NaT,None,36.5874,59.1813871,NaN,None
4,5,116082025923154031849,1,1,7,2025-09-23 15:43:55.070,14040701,1154355,25239,214,...,الکترونیک,NaN,None,None,NaT,None,None,None,NaN,None


In [175]:

tbl_FactContract.select(_.ShamsiYearMonth).limit(5).execute()

,ShamsiYearMonth
0,140008
1,140103
2,140104
3,140105
4,140106
